# 02 · Chuẩn bị dữ liệu và chống rò rỉ

**Mục tiêu.** Biến dữ liệu thô thành bảng đặc trưng mà mô hình có thể dùng **lúc 7h sáng**, trước khi ca làm việc bắt đầu, theo cách tái lập được và không để thông tin của tương lai lọt vào.

| Mục | Nội dung | Có học tham số từ dữ liệu? |
|---|---|---|
| 1 | Làm sạch nhãn `department` | Không (quy tắc cố định) |
| 2 | Đặc trưng trễ (lịch sử của từng tổ-bộ phận) và kiểm tra nhân quả | Không |
| 3 | Tách train/test theo ngày | Không |
| 4 | Hai kịch bản đặc trưng A/B | Không |
| 5 | `ColumnTransformer`: điền khuyết, `log1p`, chuẩn hoá, one-hot; kiểm tra chỉ học từ train | **Có**, nên nằm trong `Pipeline` |

Nguyên tắc chung: bước nào **không** học tham số (quy tắc cố định, như sửa chính tả) thì làm một lần trên toàn bộ dữ liệu; bước nào **có** học tham số (median, trung bình, độ lệch chuẩn, danh sách hạng mục) thì đặt trong `Pipeline` để mỗi lần huấn luyện chỉ học từ phần train của lần đó.

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # chạy được từ notebooks/ hoặc từ gốc repo
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)
warnings.filterwarnings("ignore", category=FutureWarning)

from config import DATA_FILE, SEED
import plots as P

import features as F
import splits as S
import pipeline as PL
import eda

## 1. Làm sạch nhãn `department`
**Vì sao.** Dữ liệu gốc có ba cách viết cho hai bộ phận: `sweing` (sai chính tả) và `finishing` có hoặc không có khoảng trắng thừa ở cuối. Nếu để nguyên, one-hot sẽ tạo ra 3 bộ phận và tách finishing thành hai nhóm giả. Việc sửa là quy tắc cố định (bỏ khoảng trắng, đổi `sweing` → `sewing`), không học gì từ dữ liệu, nên làm trước khi tách train/test là an toàn.

In [2]:
raw = F.load_raw(DATA_FILE)
clean = F.clean(raw)
print("Trước:", raw["department"].value_counts().to_dict())
print("Sau  :", clean["department"].value_counts().to_dict())

Trước: {'sweing': 691, 'finishing ': 257, 'finishing': 249}
Sau  : {'sewing': 691, 'finishing': 506}


**Nhận xét.**
- Trước khi sửa có 3 giá trị: `sweing` 691, `finishing ` 257 (thừa khoảng trắng), `finishing` 249.
- Sau khi sửa còn đúng 2 bộ phận: sewing 691 và finishing 506 = 257 + 249. Tổng 1.197 dòng giữ nguyên, không dòng nào bị mất.
- Hàm `clean` cũng có `assert` dừng chương trình nếu còn nhãn lạ, và kiểm tra không có hai dòng trùng (ngày, tổ, bộ phận).

## 2. Đặc trưng trễ
**Vì sao cần.** Lúc 7h sáng chưa biết năng suất hôm nay, nhưng đã biết kết quả **những ngày trước** của từng tổ. EDA cho thấy trượt có quán tính: lần trước trượt thì lần này trượt khoảng 51%, lần trước đạt thì chỉ khoảng 18%. Đặc trưng trễ đưa thông tin đó vào mô hình.

**Cách làm.** Với mỗi cặp (`team`, `department`), dùng **dòng liền trước của chính cặp đó** (ngày làm việc gần nhất mà cặp đó có dữ liệu):

| Đặc trưng | Ý nghĩa |
|---|---|
| `lag1_miss` | Lần làm việc trước có trượt không (0/1) |
| `lag1_gap` | Chênh lệch năng suất so với định mức ở lần trước (âm là hụt) |
| `roll5_gap_mean` | Trung bình chênh lệch của tối đa 5 lần trước |
| `roll5_miss_rate` | Tỷ lệ trượt trong tối đa 5 lần trước |
| `is_first_obs` | 1 nếu là dòng đầu tiên của cặp (chưa có lịch sử) |

**Vì sao theo cặp tổ-bộ phận, không theo tổ.** Sewing và finishing của cùng một tổ là hai dây chuyền khác nhau, tỷ lệ trượt rất khác nhau (ví dụ tổ 6: sewing 12%, finishing 69%). Gộp lại sẽ trộn lịch sử của hai dây chuyền.

**Vì sao `shift(1)`.** Dịch xuống một dòng để giá trị của chính ngày hôm nay không bao giờ nằm trong đặc trưng của hôm nay.

**Dòng đầu của mỗi cặp** không có lịch sử nên để trống (NaN). Pipeline điền median (học trên train), và cờ `is_first_obs = 1` báo cho mô hình biết đây là giá trị điền chứ không phải lịch sử thật.

In [3]:
df = F.build_dataset(DATA_FILE)       # đã chạy assert bên trong
one = df[(df.team == 8) & (df.department == "finishing")][["date", "gap", "miss"] + F.LAG_COLS + F.FLAG_COLS].head(10).copy()
one.insert(1, "so_ngay_tu_lan_truoc", one["date"].diff().dt.days)   # chỉ để minh hoạ, không phải đặc trưng của mô hình
one

,date,so_ngay_tu_lan_truoc,gap,miss,lag1_miss,lag1_gap,roll5_gap_mean,roll5_miss_rate,is_first_obs
11,2015-01-01,NaN,-0.073333,1,NaN,NaN,NaN,NaN,1
31,2015-01-03,2.0,-0.171667,1,1.0,-0.073333,-0.073333,1.0,0
52,2015-01-04,1.0,-0.117292,1,1.0,-0.171667,-0.122500,1.0,0
74,2015-01-05,1.0,-0.135417,1,1.0,-0.117292,-0.120764,1.0,0
95,2015-01-06,1.0,-0.118939,1,1.0,-0.135417,-0.124427,1.0,0
118,2015-01-07,1.0,-0.118939,1,1.0,-0.118939,-0.123330,1.0,0
139,2015-01-08,1.0,-0.064015,1,1.0,-0.118939,-0.132451,1.0,0
161,2015-01-10,2.0,-0.171117,1,1.0,-0.064015,-0.110920,1.0,0
184,2015-01-11,1.0,-0.146402,1,1.0,-0.171117,-0.121686,1.0,0
207,2015-01-12,1.0,-0.075000,1,1.0,-0.146402,-0.123883,1.0,0


**Nhận xét (tổ 8, finishing).**
- Đây là cặp trượt liên tục: cả 10 ngày đầu đều `miss = 1`, nên từ dòng thứ hai trở đi `lag1_miss = 1` và `roll5_miss_rate = 1`. Ví dụ này cho thấy rõ quán tính trượt mà đặc trưng trễ nắm bắt.
- `lag1_gap` của mỗi dòng đúng bằng `gap` của dòng ngay trên. Ví dụ ngày 03/01 có `lag1_gap = −0,0733`, chính là `gap` của ngày 01/01. Vậy đặc trưng chỉ nhìn về quá khứ.
- `roll5_gap_mean` mở rộng dần: ngày 03/01 là trung bình 1 lần trước, ngày 04/01 là trung bình 2 lần trước (−0,1225 = trung bình của −0,0733 và −0,1717), từ dòng thứ sáu trở đi mới đủ 5 lần.
- Dòng đầu (01/01) có mọi đặc trưng trễ là NaN và `is_first_obs = 1`.
- Cột `so_ngay_tu_lan_truoc` cho thấy "lần trước" có lúc cách 2 ngày lịch (01/01 → 03/01, 08/01 → 10/01) vì nhà máy nghỉ thứ Sáu. Đặc trưng trễ đếm theo **ngày làm việc của cặp**, không theo ngày lịch.

### Chứng minh đặc trưng trễ không rò rỉ
Hai kiểm tra tự động (`assert`): (a) dòng đầu của mỗi cặp có mọi đặc trưng trễ là NaN;
(b) **kiểm tra nhân quả**: thay năng suất từ ngày *d* trở đi bằng nhiễu ngẫu nhiên thì đặc trưng trễ của mọi dòng có ngày ≤ *d* không đổi.

In [4]:
F.assert_lags_first_day_nan(df)
F.assert_lags_causal(df)
first = ~df.duplicated(F.KEY, keep="first")
print("Số cặp (team, department):", first.sum(), "| dòng đầu có lag NaN:", int(df.loc[first, F.LAG_COLS].isna().all(axis=1).sum()))
print("OK: hai assert đều qua")

Số cặp (team, department): 24 | dòng đầu có lag NaN: 24
OK: hai assert đều qua


**Nhận xét.**
- Cả 24 cặp (12 tổ × 2 bộ phận) đều có dòng đầu với mọi đặc trưng trễ là NaN, đúng như mong đợi.
- Kiểm tra nhân quả là kiểm tra mạnh: thay năng suất **từ ngày d trở đi** (gồm cả ngày d) bằng số ngẫu nhiên, đặc trưng trễ của mọi dòng đến hết ngày d vẫn y nguyên. Nghĩa là đặc trưng không dùng thông tin của tương lai, kể cả của chính ngày hôm đó. Kiểm tra chạy với 5 ngày d chọn ngẫu nhiên (hạt giống 42).
- Đặc trưng trễ được tính một lần trên toàn bộ dữ liệu, trước khi tách train/test. Điều này hợp lệ vì nó chỉ nhìn về quá khứ: dòng test được dùng lịch sử từ train và từ các ngày test trước đó, giống hệt khi vận hành thật.

### Hai giới hạn của đặc trưng trễ
Đặc trưng trễ đếm theo **dòng** của cặp, nên có hai điểm cần biết: (a) "lần trước" có thể cách nhiều ngày lịch nếu cặp vắng dữ liệu, và (b) ở đầu chuỗi, cửa sổ 5 lần chưa đầy.

In [5]:
gap_days = df.groupby(F.KEY, sort=False)["date"].diff().dt.days.dropna()
n_prev = df.groupby(F.KEY, sort=False).cumcount()          # số dòng trước đó của cùng cặp
print("(a) Số ngày lịch giữa hai lần liên tiếp của cùng cặp:")
print(gap_days.describe().round(2).to_string())
print(f"    từ 4 ngày trở lên: {int((gap_days >= 4).sum())} / {len(gap_days)} lần; từ 7 ngày trở lên: {int((gap_days >= 7).sum())}")
print(f"(b) Dòng có ít hơn 5 lần trước (cửa sổ roll5 chưa đầy): {int((n_prev < 5).sum())} / {len(df)} "
      f"({(n_prev < 5).mean():.1%}); trong đó dòng đầu (không có lịch sử): {int((n_prev == 0).sum())}")

(a) Số ngày lịch giữa hai lần liên tiếp của cùng cặp:
count    1173.00
mean        1.38
std         1.10
min         1.00
25%         1.00
50%         1.00
75%         1.00
max        16.00
    từ 4 ngày trở lên: 36 / 1173 lần; từ 7 ngày trở lên: 10
(b) Dòng có ít hơn 5 lần trước (cửa sổ roll5 chưa đầy): 120 / 1197 (10.0%); trong đó dòng đầu (không có lịch sử): 24


**Nhận xét.**
- **(a) Khoảng cách thời gian không được mã hoá.** Phần lớn "lần trước" chỉ cách 1 ngày (trung vị 1), nhưng có những lần cách từ 4 ngày trở lên, dài nhất 16 ngày. Mô hình không biết `lag1_miss` là chuyện của hôm qua hay của hai tuần trước. Cách cải tiến có thể là thêm đặc trưng "số ngày từ lần trước", nhưng bài chưa làm.
- **(b) Cửa sổ 5 lần chưa đầy ở đầu chuỗi.** Khoảng 10% dòng có ít hơn 5 lần trước, nên `roll5_*` được tính trên ít quan sát hơn và nhiễu hơn. Ở dòng thứ hai của mỗi cặp, `roll5_*` trùng hẳn với `lag1_*` (xem ví dụ tổ 8 ở trên). Các dòng này đều nằm ở đầu giai đoạn train, nên không ảnh hưởng tới test.
- Cả hai điểm là giới hạn đã biết, không phải lỗi. Nên nhắc trong phần giới hạn của báo cáo.

## 3. Tách train/test theo ngày và kiểm tra không chung ngày
**Vì sao tách theo ngày, không tách ngẫu nhiên.** Mô hình sẽ được dùng để dự báo **ngày mai** từ dữ liệu **đến hôm nay**. Tách ngẫu nhiên sẽ cho mô hình học từ các ngày sau để đoán các ngày trước, và các tổ khác nhau trong cùng một ngày (có chung điều kiện, ví dụ cùng mã hàng) sẽ nằm ở cả hai phía. Kết quả sẽ lạc quan hơn thực tế. Thí nghiệm định lượng mức lạc quan này nằm ở `03_validation`.

**Vì sao mốc 26/02/2015.** Lấy khoảng 20% số ngày cuối làm test (12/59 ngày), phần còn lại để huấn luyện và dò siêu tham số. Test chỉ dùng **một lần** ở cuối để đánh giá.

In [6]:
train, test = S.split_train_test(df)
S.assert_no_date_overlap(train, test)
print("Dòng chưa có lịch sử (is_first_obs = 1): train", int(train.is_first_obs.sum()), "| test", int(test.is_first_obs.sum()))
pd.DataFrame({
    "tap": ["train", "test"],
    "tu_ngay": [train.date.min().date(), test.date.min().date()],
    "den_ngay": [train.date.max().date(), test.date.max().date()],
    "n_ngay": [train.date.nunique(), test.date.nunique()],
    "n_dong": [len(train), len(test)],
    "ty_le_miss": [train.miss.mean().round(4), test.miss.mean().round(4)],
})

Dòng chưa có lịch sử (is_first_obs = 1): train 24 | test 0


,tap,tu_ngay,den_ngay,n_ngay,n_dong,ty_le_miss
0,train,2015-01-01,2015-02-25,47,943,0.2694
1,test,2015-02-26,2015-03-11,12,254,0.2677


**Nhận xét.**
- Train 47 ngày (943 dòng), test 12 ngày (254 dòng): khoảng 80/20 theo số ngày và 79/21 theo số dòng. Không ngày nào nằm ở cả hai tập, và mọi ngày test đứng sau mọi ngày train.
- Tỷ lệ trượt gần như bằng nhau (26,9% so với 26,8%), nên việc tách theo thời gian không làm lệch tỷ lệ lớp. Tuy vậy EDA (hình chuỗi thời gian) cho thấy test bắt đầu ngay sau giai đoạn trượt cao giữa tháng 2, nên điều kiện test không hẳn giống train.
- Cả 24 dòng `is_first_obs = 1` đều ở train; test không có dòng nào, vì mọi cặp đều đã có lịch sử trong train. Như vậy giá trị median điền cho dòng đầu không ảnh hưởng gì tới test.

## 4. Hai kịch bản đặc trưng
**Vì sao có hai kịch bản.** Quyết định được ra lúc 7h, nên chỉ được dùng biến **đã biết lúc đó**. Với `incentive`, `over_time` và `no_of_style_change`, nguồn dữ liệu không nói rõ được ghi lúc nào: thưởng có thể trả theo kết quả trong ngày, tăng ca có thể phát sinh khi tổ đang chậm. Nếu đúng vậy, chúng chứa một phần đáp án (rò rỉ).
- **Kịch bản A (chính):** chỉ dùng biến chắc chắn biết đầu ca. Đây là kịch bản dùng cho mọi kết luận.
- **Kịch bản B (độ nhạy):** A cộng thêm ba biến trên, để xem kết quả thay đổi bao nhiêu nếu được phép dùng chúng.

Bảng dưới liệt kê **mọi cột** của dữ liệu, cột nào được dùng ở kịch bản nào và vì sao. Phần đánh dấu A/B lấy trực tiếp từ cấu hình `PL.SCENARIOS`, nên luôn khớp với code.

In [7]:
REASON = {
    "department": ("phân loại", "Biết trước; finishing trượt gấp 2,4 lần sewing"),
    "day": ("phân loại", "Biết trước (thứ trong tuần)"),
    "team": ("phân loại", "Biết trước; mã tổ là nhãn, không có thứ tự, nên one-hot"),
    "targeted_productivity": ("số", "Định mức đặt trước ca"),
    "smv": ("số", "Thời gian chuẩn của mã hàng, biết khi lên kế hoạch"),
    "no_of_workers": ("số", "Số công nhân theo kế hoạch nhân sự"),
    "week_of_month": ("số", "Tạo lại từ date, thay cho quarter"),
    "wip": ("wip", "Bán thành phẩm tồn đầu ca; finishing không ghi nên điền 0"),
    "lag1_miss": ("trễ", "Lịch sử của cặp tổ-bộ phận, chỉ dùng quá khứ"),
    "lag1_gap": ("trễ", "Lịch sử của cặp tổ-bộ phận, chỉ dùng quá khứ"),
    "roll5_gap_mean": ("trễ", "Lịch sử của cặp tổ-bộ phận, chỉ dùng quá khứ"),
    "roll5_miss_rate": ("trễ", "Lịch sử của cặp tổ-bộ phận, chỉ dùng quá khứ"),
    "is_first_obs": ("cờ", "Báo dòng chưa có lịch sử (đặc trưng trễ là giá trị điền)"),
    "incentive": ("số", "Không rõ thời điểm ghi; có thể trả theo kết quả, rủi ro rò rỉ lớn nhất"),
    "over_time": ("số", "Không rõ thời điểm ghi; có thể phát sinh khi tổ đang chậm"),
    "no_of_style_change": ("số", "Nhiều khả năng theo kế hoạch nhưng chưa chắc"),
    "idle_time": ("loại", "Phát sinh trong ngày, không thể biết lúc 7h"),
    "idle_men": ("loại", "Phát sinh trong ngày, không thể biết lúc 7h"),
    "quarter": ("loại", "Thực chất là tuần trong tháng; đã thay bằng week_of_month"),
    "date": ("loại", "Chỉ dùng để sắp xếp, tạo đặc trưng và tách train/test"),
    "actual_productivity": ("loại", "Kết quả cuối ngày; là nguồn của nhãn"),
    "gap": ("loại", "Kết quả cuối ngày; chỉ dùng để tạo nhãn và đặc trưng trễ"),
    "miss": ("loại", "Biến mục tiêu"),
}
in_A, in_B = set(PL.SCENARIOS["A"].columns), set(PL.SCENARIOS["B"].columns)
assert (in_A | in_B) <= set(REASON), "Có cột trong kịch bản chưa được giải thích"
scen = pd.DataFrame([
    {"cot": c, "nhom": g, "A": "✓" if c in in_A else "", "B": "✓" if c in in_B else "", "vi_sao": r}
    for c, (g, r) in REASON.items()
])
print(f"Kịch bản A: {len(in_A)} cột đầu vào | Kịch bản B: {len(in_B)} cột đầu vào")
scen

Kịch bản A: 13 cột đầu vào | Kịch bản B: 16 cột đầu vào


,cot,nhom,A,B,vi_sao
0,department,phân loại,✓,✓,"Biết trước; finishing trượt gấp 2,4 lần sewing"
1,day,phân loại,✓,✓,Biết trước (thứ trong tuần)
2,team,phân loại,✓,✓,"Biết trước; mã tổ là nhãn, không có thứ tự, nên one-hot"
3,targeted_productivity,số,✓,✓,Định mức đặt trước ca
4,smv,số,✓,✓,"Thời gian chuẩn của mã hàng, biết khi lên kế hoạch"
5,no_of_workers,số,✓,✓,Số công nhân theo kế hoạch nhân sự
6,week_of_month,số,✓,✓,"Tạo lại từ date, thay cho quarter"
7,wip,wip,✓,✓,Bán thành phẩm tồn đầu ca; finishing không ghi nên điền 0
8,lag1_miss,trễ,✓,✓,"Lịch sử của cặp tổ-bộ phận, chỉ dùng quá khứ"
9,lag1_gap,trễ,✓,✓,"Lịch sử của cặp tổ-bộ phận, chỉ dùng quá khứ"


**Nhận xét.**
- Kịch bản A dùng 13 cột đầu vào: 3 phân loại, 4 số, `wip`, 4 đặc trưng trễ và 1 cờ. Kịch bản B dùng 16 cột (thêm `incentive`, `over_time`, `no_of_style_change`).
- `idle_time`, `idle_men` bị loại ở **cả hai** kịch bản vì chắc chắn phát sinh trong ngày. Đây là ví dụ rõ nhất của rò rỉ theo thời điểm: chúng có thể dự báo tốt, nhưng lúc 7h không ai biết giá trị của chúng.
- `actual_productivity` và `gap` không bao giờ là đầu vào trực tiếp; chúng chỉ đi vào mô hình qua đặc trưng trễ (dịch một ngày).
- Nếu kết quả B tốt hơn A nhiều, không thể kết luận mô hình B tốt hơn: có thể chỉ vì B "nhìn trộm" kết quả qua `incentive`. Đó là lý do mọi kết luận chính đều dựa trên A.

## 5. `ColumnTransformer`
Đây là các bước **có học tham số từ dữ liệu**, nên đều nằm trong `Pipeline`. Khi huấn luyện (kể cả trong từng fold của kiểm định chéo), pipeline chỉ `fit` trên phần train của lần đó rồi áp dụng nguyên tham số ấy cho phần còn lại.

| Nhóm cột | Các bước | Vì sao |
|---|---|---|
| Số (`targeted_productivity`, `smv`, `no_of_workers`, `week_of_month`) | điền median → chuẩn hoá | Hiện không có giá trị thiếu; median là chốt an toàn nếu dữ liệu mới bị thiếu. Chuẩn hoá đưa các biến về cùng thang đo, cần cho Logistic, SVM và MLP |
| `wip` | điền 0 → `log1p` → chuẩn hoá | `wip` chỉ thiếu ở finishing, nơi không giữ bán thành phẩm, nên 0 đúng nghĩa vật lý (điền median sẽ gán số liệu của sewing cho finishing). `log1p` nén đuôi phải dài (độ lệch 9,74); xem hình trước và sau ở `01_eda` |
| Đặc trưng trễ | điền median → chuẩn hoá | Chỉ thiếu ở dòng đầu của mỗi cặp; median của train là giá trị "điển hình", và cờ `is_first_obs` báo đây là giá trị điền |
| `is_first_obs` | giữ nguyên (0/1) | Đã là cờ nhị phân, không cần chuẩn hoá |
| Phân loại (`department`, `day`, `team`) | one-hot | Là nhãn không có thứ tự (tổ 12 không "lớn hơn" tổ 1) |

**Không loại ngoại lai.** Ngoại lai trong dữ liệu này phần lớn là vận hành thật (ngày trượt nặng, định mức thấp, `wip` lớn). Thay vì loại dòng, bài dùng `log1p` và chuẩn hoá để giảm ảnh hưởng, và cây/Random Forest vốn không nhạy với ngoại lai.

**`week_of_month` là biến số** (1–5) chứ không one-hot, vì chỉ có 5 mức và có thứ tự tự nhiên trong tháng.

In [8]:
pipe = PL.build_model_pipeline("Logistic", "A")
X_tr = train[PL.SCENARIOS["A"].columns]
pipe.fit(X_tr, train["miss"])
names = PL.feature_names(pipe.named_steps["prep"])
print(len(names), "đặc trưng sau biến đổi")
print(names)
assert names.count("is_first_obs") == 1 and not any(n.startswith(("wip_missing", "lag_missing")) for n in names)

30 đặc trưng sau biến đổi
['targeted_productivity', 'smv', 'no_of_workers', 'week_of_month', 'log1p(wip)', 'lag1_miss', 'lag1_gap', 'roll5_gap_mean', 'roll5_miss_rate', 'is_first_obs', 'department_finishing', 'department_sewing', 'day_Monday', 'day_Saturday', 'day_Sunday', 'day_Thursday', 'day_Tuesday', 'day_Wednesday', 'team_1', 'team_2', 'team_3', 'team_4', 'team_5', 'team_6', 'team_7', 'team_8', 'team_9', 'team_10', 'team_11', 'team_12']


**Nhận xét.**
- 13 cột đầu vào thành **30 đặc trưng**: 4 số + `log1p(wip)` + 4 đặc trưng trễ + 1 cờ + 2 cột `department` + 6 cột `day` (không có thứ Sáu) + 12 cột `team`.
- `assert` ở cuối ô xác nhận chỉ có đúng một cờ `is_first_obs`, không còn cờ thiếu trùng lặp như `wip_missing` (cờ này sẽ trùng 100% với `department_finishing`).
- **Lưu ý khi đọc hệ số:** one-hot giữ đủ mọi hạng mục (không bỏ cột nào), nên `department_finishing` và `department_sewing` luôn cộng lại bằng 1, tương tự với `day` và `team`. Với Logistic, hệ số của từng cột trong một nhóm không có nghĩa riêng; chỉ **chênh lệch** giữa các cột trong cùng nhóm mới có nghĩa. Cây và Random Forest không bị ảnh hưởng. Bài giữ nguyên cách mã hoá này để không làm thay đổi kết quả đã chạy, nhưng cần nhớ khi đọc hệ số ở `06_interpret`.

### Kiểm tra pipeline chỉ học tham số từ train
Pipeline vừa được `fit` trên train. Ô dưới kiểm tra **mọi** tham số đã học (trung bình và độ lệch chuẩn của bộ chuẩn hoá, median của bộ điền khuyết, danh sách hạng mục của one-hot) đúng là thống kê của train và khác thống kê của toàn bộ dữ liệu.

Đây là kiểm tra cơ bản, xác nhận các bước được nối đúng chỗ. Rủi ro rò rỉ lớn hơn nằm ở kiểm định chéo (mỗi fold phải `fit` lại từ đầu), và được xử lý ở `03_validation`: `GridSearchCV` nhân bản toàn bộ pipeline cho từng fold.

In [9]:
sc = PL.SCENARIOS["A"]
ct = pipe.named_steps["prep"].named_transformers_
X_te = test[sc.columns]

def compare(name, learned, train_stat, all_stat):
    assert np.allclose(learned, train_stat), f"{name}: không khớp train"
    return pd.DataFrame({"nhom": name, "hoc_duoc (pipeline)": learned, "thong_ke_train": train_stat,
                         "thong_ke_toan_bo": all_stat, "khac_toan_bo": ~np.isclose(learned, all_stat)})

wip_tr, wip_all = np.log1p(train["wip"].fillna(0)), np.log1p(df["wip"].fillna(0))
lag_tr_filled = train[sc.lag].fillna(train[sc.lag].median())
rows = [
    compare("số: mean", ct["num"].named_steps["scale"].mean_, train[sc.numeric].mean().to_numpy(), df[sc.numeric].mean().to_numpy()),
    compare("số: std", ct["num"].named_steps["scale"].scale_, train[sc.numeric].std(ddof=0).to_numpy(), df[sc.numeric].std(ddof=0).to_numpy()),
    compare("wip: mean log1p", ct["wip"].named_steps["scale"].mean_, [wip_tr.mean()], [wip_all.mean()]),
    compare("wip: std log1p", ct["wip"].named_steps["scale"].scale_, [wip_tr.std(ddof=0)], [wip_all.std(ddof=0)]),
    compare("trễ: median", ct["lag"].named_steps["impute"].statistics_, train[sc.lag].median().to_numpy(), df[sc.lag].median().to_numpy()),
    compare("trễ: mean", ct["lag"].named_steps["scale"].mean_, lag_tr_filled.mean().to_numpy(), df[sc.lag].fillna(df[sc.lag].median()).mean().to_numpy()),
]
idx = sc.numeric * 2 + ["wip"] * 2 + sc.lag * 2
check = pd.concat(rows).set_axis(idx)
assert check["khac_toan_bo"].any(), "Tham số học được trùng với toàn bộ dữ liệu: không phân biệt được"

# wip: bước điền khuyết là hằng 0, không học gì
assert ct["wip"].named_steps["impute"].statistics_[0] == 0.0

# one-hot: hạng mục học được đúng bằng các giá trị xuất hiện trong train
enc = ct["cat"]
for col, cats in zip(sc.categorical, enc.categories_):
    assert set(cats) == set(train[col].unique()), f"one-hot {col}: hạng mục khác train"
print("Hạng mục one-hot:", {c: [str(x) for x in k] for c, k in zip(sc.categorical, enc.categories_)})
print("Hạng mục có ở test mà không có ở train:",
      {c: sorted(set(test[c]) - set(train[c])) for c in sc.categorical})

# áp dụng cho test: dùng đúng tham số của train
Z = pipe.named_steps["prep"].transform(X_te)
z_expected = (X_te["smv"].to_numpy() - train["smv"].mean()) / train["smv"].std(ddof=0)
assert np.allclose(Z[:, sc.numeric.index("smv")], z_expected)
print("OK: mọi tham số học từ train; test được biến đổi bằng tham số của train")
check

Hạng mục one-hot: {'department': ['finishing', 'sewing'], 'day': ['Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday'], 'team': ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']}
Hạng mục có ở test mà không có ở train: {'department': [], 'day': [], 'team': []}
OK: mọi tham số học từ train; test được biến đổi bằng tham số của train


,nhom,hoc_duoc (pipeline),thong_ke_train,thong_ke_toan_bo,khac_toan_bo
targeted_productivity,số: mean,0.732895,0.732895,0.729632,True
smv,số: mean,15.244560,15.244560,15.062172,True
no_of_workers,số: mean,35.084836,35.084836,34.609858,True
week_of_month,số: mean,2.546129,2.546129,2.399332,True
targeted_productivity,số: std,0.098526,0.098526,0.097850,True
smv,số: std,11.018864,11.018864,10.938647,True
no_of_workers,số: std,22.210891,22.210891,22.188413,True
week_of_month,số: std,1.213106,1.213106,1.215027,True
wip,wip: mean log1p,4.015159,4.015159,3.941583,True
wip,wip: std log1p,3.424159,3.424159,3.415976,True


**Nhận xét.**
- Ở mọi nhóm (biến số, `wip`, đặc trưng trễ), tham số pipeline học được trùng khớp với thống kê của **train**. Hầu hết khác thống kê của toàn bộ dữ liệu (cột `khac_toan_bo` = True), tức là nếu pipeline vô tình học trên cả test thì kiểm tra này sẽ phát hiện ra.
- Một vài dòng có `khac_toan_bo = False` (ví dụ median của `lag1_miss` bằng 0 ở cả train lẫn toàn bộ). Đó là trùng hợp tự nhiên của median trên biến 0/1, không phải rò rỉ.
- `wip` điền bằng hằng 0, không học gì từ dữ liệu.
- One-hot học đúng các hạng mục có trong train. Test không có hạng mục mới; nếu có, `handle_unknown="ignore"` sẽ mã hoá thành toàn 0 thay vì báo lỗi.
- Khi biến đổi test, `smv` được chuẩn hoá bằng trung bình và độ lệch chuẩn của train, đúng như khi vận hành thật (lúc dự báo chưa biết thống kê của tương lai).

## 6. Tổng kết

| Quyết định | Cách làm | Lý do chính |
|---|---|---|
| Làm sạch nhãn | `sweing` → `sewing`, bỏ khoảng trắng | Tránh tạo bộ phận giả khi one-hot |
| Đặc trưng trễ | Theo cặp tổ-bộ phận, `shift(1)`, cửa sổ 5 lần | Trượt có quán tính; chỉ dùng quá khứ, đã kiểm tra nhân quả |
| Tách dữ liệu | Theo ngày, test từ 26/02/2015 | Mô phỏng dự báo ngày mai từ dữ liệu đến hôm nay |
| Chọn biến | Kịch bản A chỉ dùng biến biết đầu ca; B để đo độ nhạy | Tránh rò rỉ theo thời điểm (`idle_*`, có thể cả `incentive`) |
| Điền khuyết | `wip` điền 0; đặc trưng trễ điền median kèm cờ | Đúng nghĩa vật lý; báo cho mô hình biết giá trị nào là điền |
| Biến đổi | `log1p(wip)`, chuẩn hoá biến số, one-hot biến phân loại | Giảm ảnh hưởng đuôi dài; cùng thang đo; nhãn không có thứ tự |
| Ngoại lai | Giữ nguyên | Phần lớn là vận hành thật, chính là thứ cần dự báo |

Giới hạn cần nêu trong báo cáo: đặc trưng trễ không biết "lần trước" cách bao lâu; cửa sổ 5 lần chưa đầy ở khoảng 10% dòng đầu chuỗi; one-hot giữ đủ hạng mục nên hệ số Logistic chỉ đọc được theo chênh lệch trong nhóm.